In [ ]:
%load_ext autoreload
%autoreload 2
import json
import os
import sys
sys.path.insert(0, os.path.abspath('../'))
import ResParTools as pt

# 3. RTP, силовое поле и тестовая МД

В этом шаге создаются явные файлы силового поля: RTP, HDB, таблица типов и недостающие параметры.
Сначала проверьте содержимое RTP; затем подготовьте и запустите GROMACS в окружении `gmx`.

In [ ]:
# Параметры модификации (заполняются генератором)
FOLDER = 'Lysine_prop'
PTM_name = 'Lysine_prop'
base_name = 'KPR'
monomer_file = 'molecules/K_prop.smiles'
OUT = os.path.abspath(os.path.join('..', 'build', 'notebooks', FOLDER))
INPUT = os.path.join(OUT, f'5_{os.path.splitext(os.path.basename(monomer_file))[0]}.sdf')
FF_OUT = os.path.abspath(os.path.join('..', 'build', 'amber14sb_mod_smi.ff'))
MD_OUT = os.path.abspath(os.path.join('..', 'build', 'md_from_smiles', FOLDER))
MD_NS = 0.01  # тестовая динамика 10 пс, кроме EM/NVT/NPT
run_gromacs_now = False

## Шаг 1. Загрузить молекулу с зарядами и типами

Имена, заряды Espaloma/RESP и типы acpype должны остаться привязаны к тем же атомам.

In [ ]:
residue = pt.load_mol(INPUT)
assert residue.GetProp('ChargeMethod') in ('Espaloma', 'RESP')
assert all(a.HasProp('AcpypeType') for a in residue.GetAtoms())
print('Используемые заряды:', residue.GetProp('ChargeMethod'))

## Шаг 2. Собрать RTP и силовое поле

`write_forcefield` пишет отдельную запись `<base_name>.rtp`, `<base_name>.hdb`, `residuetypes.dat`,
`atomtypes.atp` и нужные дополнения `ffnonbonded.itp`. Выход находится в `build/`, исходное
силовое поле не редактируется.

In [ ]:
rtp_path, explicit = pt.write_forcefield(residue, FF_OUT)
residue_sdf = pt.save_mol(residue, os.path.join(OUT, f'6_{os.path.splitext(os.path.basename(monomer_file))[0]}'))
print('RTP:', rtp_path)
print('Поле:', FF_OUT)
print('Явные bonded-параметры:', {key: len(value) for key, value in explicit.items()})
with open(rtp_path) as f:
    print(f.read())
print('Файлы .ff:', sorted(os.listdir(FF_OUT)))

## Шаг 3. Подготовить тест системы и выполнить проверку GROMACS

Входная модель — ACE–остаток–NME, чтобы проверить связь модифицированного остатка с соседями.
`pdb2gmx`, `grompp`, EM и 10-пс MD запускаются через пакетную функцию. По умолчанию создаётся
папка задания; включите `run_gromacs_now` для запуска локально или используйте сгенерированный
SLURM-скрипт на кластере.

In [ ]:
monomer = pt.open_mol(monomer_file)
job = pt.write_validate_job(residue, monomer, FF_OUT, MD_OUT, md_ns=MD_NS)
print('Проверочная система:', job)

if run_gromacs_now:
    report = pt.run_validate(job, run='local', n_threads=4)
    print(json.dumps(report, ensure_ascii=False, indent=2))
else:
    print('Локально: включите run_gromacs_now в окружении gmx и выполните ячейку.')
    print('Кластер: pt.run_validate(job, run="slurm", slurm={...}); затем sbatch run_validate.sbatch.')

## Результат

Силовое поле: `build/amber14sb_mod_smi.ff/<base_name>.rtp` и согласованные файлы `.ff`.
Отчёт и входные файлы теста: `build/md_from_smiles/<модификация>/`. Успешный запуск GROMACS не заменяет
проверку имён атомов, зарядов, RTP и границ `-C`/`+N`.